Description

## Initialization

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log, ceil
import shutil
from itertools import chain

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.integrate import quad as get_integral

from data_processing.arc_paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting.get_histogram import get_psd_energy_histogram
from data_processing.processing.slice_fitting.scan_histogram_slices import scan_histogram_slices
from data_processing.processing.slice_fitting.helpers import find_failed_slices
from data_processing.processing.slice_fitting.bimodal_fitting import get_bimodal_fit_guess, get_bimodal_fit
from data_processing.processing.figure_of_merit import bimodal, gaussian
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing.helpers import (
    stop,
    get_input_with_default,
    input_experiment_ids,
    get_midpoints_from_min_max_series,
    get_midpoints_from_bins
)
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)

In [ ]:
print("Enter E-cell experiment IDs")
ecell_experiment_ids = input_experiment_ids()
print()
print("Enter beam-loading experiment IDs")
beam_experiment_ids = input_experiment_ids()
all_experiment_ids = list(zip(beam_experiment_ids, ecell_experiment_ids))

In [ ]:
LinkableDataset = dict[str, dict[ExperimentDataKey, Any] | str]
LinkedDatasets = tuple[LinkableDataset, LinkableDataset]
all_experiment_data: list[LinkedDatasets] = [
    (
        {"exp_id": beam_exp_id, "data": {}},
        {"exp_id": ecell_exp_id, "data": {}}
    )
    for beam_exp_id, ecell_exp_id in all_experiment_ids
]

In [ ]:
calibrated_energy_column = DetectorDataframeColumn.RECALIBRATED_ENERGY

## Data Loading and Initial Processing

### Neutron Data Processing

In [ ]:
# Data Loading
for linked_dataset in all_experiment_data:
    for dataset in linked_dataset:
        exp_id = dataset['exp_id']
        exp_data = dataset['data']
        detector_df = load_psd(exp_id)
        exp_data[ExperimentDataKey.UNCLASSIFIED] = detector_df

In [ ]:
# Express timetags in hours elapsed
for linked_dataset in all_experiment_data:
    for dataset in linked_dataset:
        exp_data = dataset['data']
        unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
        unclassified_df = calculate_timetag_hours(unclassified_df)
        exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for linked_dataset in all_experiment_data:
    for dataset in linked_dataset:
        exp_data = dataset['data']
        unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
        unclassified_df = recalibrate(unclassified_df, Detector.ZERO)
        exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 5e-3
# overall_settings['scan_idx'] = f"({start_scan_idx}, {end_scan_idx})"
# overall_settings['energy_width'] = energy_width
for linked_dataset in all_experiment_data:
    for dataset in linked_dataset:
        exp_data = dataset['data']
        psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
        Z, xe, ye = get_psd_energy_histogram(
            psd_report,
            calibrated_energy_column,
            energy_width=energy_width
        )
        exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
        exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
        exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
        exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
make_single_slice_plots = get_input_with_default(
    "Do you want to make test plots on a slice? [y/n, or press Enter for no]",
    False,
    bool
)
slice_idx = None
if make_single_slice_plots:
    slice_idx = get_input_with_default(
"""\
Enter slice index to plot
Press Enter for default (30)
""",
        30,
        int
    )

In [ ]:
# get single slice, fit guess, and fit data
if slice_idx is not None:
    for linked_dataset in all_experiment_data:
        for dataset in linked_dataset:
            exp_id = dataset['exp_id']
            exp_data = dataset['data']
            Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
            ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]

            slice = Z[slice_idx, :]
            exp_data['slice'] = slice
            
            fit_guess = get_bimodal_fit_guess(ye, slice)
            exp_data['fit_guess'] = fit_guess
            print(fit_guess)
            
            psd_bin_mids = get_midpoints_from_bins(ye)
            exp_data['psd_bin_mids'] = psd_bin_mids
            
            gauss_gamma_params, gauss_n_params, _ = get_bimodal_fit(
                psd_bin_mids, slice, guess=fit_guess
            )
            exp_data['gamma_fit_params'] = gauss_gamma_params
            exp_data['neutron_fit_params'] = gauss_n_params

In [ ]:
if slice_idx is not None:
    bimodal_x = np.linspace(0, 0.5, 1000)

    for linked_dataset in all_experiment_data:
        beam_dataset, ecell_dataset = linked_dataset
        dataset_list = [("Beam", beam_dataset), ("Ecell", ecell_dataset)]
        fig = plt.figure(figsize=(16, 8))

        for i, dataset_list_item in enumerate(dataset_list):
            exp_type, dataset = dataset_list_item
            exp_id = dataset['exp_id']
            exp_data = dataset['data']
            fit_guess = exp_data['fit_guess']
            psd_bin_mids = exp_data['psd_bin_mids']
            slice = exp_data['slice']

            bimodal_y = bimodal(
                bimodal_x,
                fit_guess.mu1,
                fit_guess.sigma1,
                fit_guess.a1,
                fit_guess.mu2,
                fit_guess.sigma2,
                fit_guess.a2
            )
            gaussian_g_y = gaussian(
                bimodal_x, fit_guess.mu1, fit_guess.sigma1, fit_guess.a1
            )
            gaussian_n_y = gaussian(
                bimodal_x, fit_guess.mu2, fit_guess.sigma2, fit_guess.a2
            )

            ax = fig.add_subplot(1, len(dataset_list), i+1)
            ax.errorbar(
                psd_bin_mids, slice, yerr=np.sqrt(slice), fmt=".g", capsize=3)
            ax.plot(bimodal_x, bimodal_y, "-m")
            ax.plot(bimodal_x, gaussian_g_y, "--r", alpha=0.2)
            ax.plot(bimodal_x, gaussian_n_y, "--b", alpha=0.2)
            ax.plot(fit_guess.mu1, fit_guess.a1, "xr")
            ax.plot(fit_guess.mu2, fit_guess.a2, "xb")
            ax.axvspan(
                fit_guess.mu1-fit_guess.sigma1,
                fit_guess.mu1+fit_guess.sigma1,
                alpha=0.2,
                color="r"
            )
            ax.axvspan(
                fit_guess.mu2-fit_guess.sigma2,
                fit_guess.mu2+fit_guess.sigma2,
                alpha=0.2,
                color="b"
            )
            ax.set_title(f"{exp_type} ({exp_id})")
    
        plt.show()

In [ ]:
if slice_idx is not None:
    bimodal_x = np.linspace(0, 0.5, 1000)

    for linked_dataset in all_experiment_data:
        beam_dataset, ecell_dataset = linked_dataset
        dataset_list = [("Beam", beam_dataset), ("Ecell", ecell_dataset)]
        fig = plt.figure(figsize=(16, 10))

        for i, dataset_list_item in enumerate(dataset_list):
            exp_type, dataset = dataset_list_item
            exp_id = dataset['exp_id']
            exp_data = dataset['data']
            fit_guess = exp_data['fit_guess']
            psd_bin_mids = exp_data['psd_bin_mids']
            slice = exp_data['slice']
            gauss_gamma_params = exp_data['gamma_fit_params']
            gauss_n_params = exp_data['neutron_fit_params']

            bimodal_y = bimodal(
                bimodal_x,
                gauss_gamma_params.mu,
                gauss_gamma_params.sigma,
                gauss_gamma_params.a,
                gauss_n_params.mu,
                gauss_n_params.sigma,
                gauss_n_params.a
            )
            bimodal_at_bins = bimodal(
                psd_bin_mids,
                gauss_gamma_params.mu,
                gauss_gamma_params.sigma,
                gauss_gamma_params.a,
                gauss_n_params.mu,
                gauss_n_params.sigma,
                gauss_n_params.a
            )
            bimodal_residuals = slice - bimodal_at_bins
            gaussian_g_y = gaussian(
                bimodal_x,
                gauss_gamma_params.mu,
                gauss_gamma_params.sigma,
                gauss_gamma_params.a
            )
            gaussian_n_y = gaussian(
                bimodal_x,
                gauss_n_params.mu,
                gauss_n_params.sigma,
                gauss_n_params.a
            )

            ax = fig.add_subplot(1, len(dataset_list), i+1)
            divider = make_axes_locatable(ax)
            resids_ax = divider.append_axes(
                "bottom", size=1, pad=0.2, sharex=ax
            )
            ax.tick_params('x', labelbottom=False)

            ax.errorbar(
                psd_bin_mids, slice, yerr=np.sqrt(slice), fmt=".g", capsize=3
            )
            ax.plot(bimodal_x, bimodal_y, "m-")
            ax.plot(bimodal_x, gaussian_g_y, "--r")
            ax.plot(bimodal_x, gaussian_n_y, "--b")
            ax.plot(fit_guess.mu1, fit_guess.a1, "xr")
            ax.plot(fit_guess.mu2, fit_guess.a2, "xb")
            ax.axvspan(
                gauss_gamma_params.mu-gauss_gamma_params.sigma,
                gauss_gamma_params.mu+gauss_gamma_params.sigma,
                alpha=0.2,
                color="r"
            )
            ax.axvspan(
                gauss_n_params.mu-gauss_n_params.sigma,
                gauss_n_params.mu+gauss_n_params.sigma,
                alpha=0.2,
                color="b"
            )
            ax.axvspan(
                fit_guess.mu1-fit_guess.sigma1,
                fit_guess.mu1+fit_guess.sigma1,
                alpha=0.2,
                color="c"
            )
            ax.axvspan(
                fit_guess.mu2-fit_guess.sigma2,
                fit_guess.mu2+fit_guess.sigma2,
                alpha=0.2,
                color="y"
            )

            resids_ax.plot(psd_bin_mids, bimodal_residuals, "og")
            resids_ax.axvline(gauss_gamma_params.mu, color="k", linestyle="--")
            resids_ax.axvline(gauss_n_params.mu, color="k", linestyle="--")
            resids_ax.axhline(0, color="k", linestyle="dotted", linewidth=1)
            resids_ax.axvspan(
                gauss_gamma_params.mu-gauss_gamma_params.sigma,
                gauss_gamma_params.mu+gauss_gamma_params.sigma,
                alpha=0.2,
                color="r"
            )
            resids_ax.axvspan(
                gauss_n_params.mu-gauss_n_params.sigma,
                gauss_n_params.mu+gauss_n_params.sigma,
                alpha=0.2,
                color="b"
            )
            ax.set_title(f"{exp_type} ({exp_id})")

        plt.show()

In [ ]:
for linked_dataset in all_experiment_data:
    for dataset in linked_dataset:
        exp_data = dataset['data']
        Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
        xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
        ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
        end_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]
        
        fit_df, _ = scan_histogram_slices(
            Z, xe, ye, fit_style="peak_finder", end_idx=end_idx
        )

        exp_data[ExperimentDataKey.FOM_RESULTS] = fit_df

In [ ]:
stride = None
plot_cols = None
start_idx = 0

make_slice_plots = get_input_with_default(
    "Do you want to make plots for all slices? [y/n, or press Enter for no]",
    False,
    bool
)

if make_slice_plots:
    stride = get_input_with_default(
"""\
Enter slice stride (how often to plot)
Press Enter for default (4)
""",
        4,
        int
    )
    plot_cols = get_input_with_default(
"""\
Enter number of plot columns to use
Press Enter for default (4)
""",
    4,
    int
    )

In [ ]:
if stride is not None and plot_cols is not None:
    bimodal_x = np.linspace(0, 0.5, 1000)

    for linked_dataset in all_experiment_data:
        beam_dataset, ecell_dataset = linked_dataset
        dataset_list = [("Beam", beam_dataset), ("Ecell", ecell_dataset)]

        for i, dataset_list_item in enumerate(dataset_list):
            exp_type, dataset = dataset_list_item
            exp_id = dataset['exp_id']
            exp_data = dataset['data']
            # psd_bin_mids = exp_data['psd_bin_mids']
            ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
            psd_bin_mids = get_midpoints_from_bins(ye)
            Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
            # gauss_gamma_params = exp_data['gamma_fit_params']
            # gauss_n_params = exp_data['neutron_fit_params']
            end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]
            fit_df = exp_data[ExperimentDataKey.FOM_RESULTS]

            slice_idxs = list(range(start_scan_idx, end_scan_idx, stride))
            plot_count = len(slice_idxs)
            plot_rows = ceil(plot_count/plot_cols)

            fig, axs = plt.subplots(
                plot_rows,
                plot_cols,
                figsize=(8*plot_cols, 8*plot_rows),
                gridspec_kw=dict(wspace=0.1, hspace=0.1),
                # sharex=True
            )
            fig.suptitle(f"{exp_type} ({exp_id})", y=0.89, fontsize="x-large")
            for fig_idx, i in enumerate(slice_idxs):
                fig_col = fig_idx % plot_cols
                fig_row = fig_idx // plot_cols
                ax = axs[fig_row, fig_col]

                divider = make_axes_locatable(ax)
                resids_ax = divider.append_axes(
                    "bottom", size=1, pad=0.2, sharex=ax
                )
                ax.tick_params('x', labelbottom=False)
                if fig_row < (plot_rows - 1):
                    resids_ax.tick_params('x', labelbottom=False)

                plot_slice = Z[i, :]
                slice_fit_params = fit_df.loc[i]
                mu1 = slice_fit_params.mu1
                sigma1 = slice_fit_params.sigma1
                A1 = slice_fit_params.a1
                mu2 = slice_fit_params.mu2
                sigma2 = slice_fit_params.sigma2
                A2 = slice_fit_params.a2
                e_min = slice_fit_params.slice_energy_min
                e_max = slice_fit_params.slice_energy_max
                e_mid = (e_max + e_min) / 2

                bimodal_y = bimodal(bimodal_x, mu1, sigma1, A1, mu2, sigma2, A2)
                gaussian_g_y = gaussian(bimodal_x, mu1, sigma1, A1)
                gaussian_n_y = gaussian(bimodal_x, mu2, sigma2, A2)
                bimodal_at_bins = bimodal(
                    psd_bin_mids,
                    mu1,
                    sigma1,
                    A1,
                    mu2,
                    sigma2,
                    A2
                )
                bimodal_residuals = plot_slice - bimodal_at_bins
            
                ax.errorbar(psd_bin_mids, plot_slice, yerr=np.sqrt(plot_slice), fmt=".g", capsize=3)
                ax.plot(bimodal_x, bimodal_y, "m-")
                ax.plot(bimodal_x, gaussian_g_y, "--r")
                ax.plot(bimodal_x, gaussian_n_y, "--b")
                ax.axvspan(
                    mu1-sigma1,
                    mu1+sigma1,
                    alpha=0.2,
                    color="r"
                )
                ax.axvspan(
                    mu2-sigma2,
                    mu2+sigma2,
                    alpha=0.2,
                    color="b"
                )
                ax.set_title(f"Slice index {i} ({e_mid*1000:.2f} keVee)")
            
                # TODO make residuals plot
                resids_ax.plot(psd_bin_mids, bimodal_residuals, "og")
                resids_ax.axvline(mu1, color="k", linestyle="--")
                resids_ax.axvline(mu2, color="k", linestyle="--")
                resids_ax.axhline(0, color="k", linestyle="dotted", linewidth=1)
                resids_ax.axvspan(
                    mu1-sigma1,
                    mu1+sigma1,
                    alpha=0.2,
                    color="r"
                )
                resids_ax.axvspan(
                    mu2-sigma2,
                    mu2+sigma2,
                    alpha=0.2,
                    color="b"
                )
            # fig.tight_layout()
            plt.show()

In [ ]:
def integrate_gaussian(mu, sigma, A):
    return get_integral(gaussian, 0, 0.5, args=(mu, sigma, A))


def integrate_gamma(row):
    return integrate_gaussian(row["mu1"], row['sigma1'], row['a1'])


def integrate_neutrons(row):
    return integrate_gaussian(row["mu2"], row['sigma2'], row['a2'])


def integration_to_ndarray(dataframe, integration_fn):
    integration = dataframe.apply(integration_fn, axis=1)
    integration = integration[
        dataframe['slice_energy_min'] > (50 * 1e-3)
    ]
    integration_ndarray = np.array(integration.tolist())
    return np.nan_to_num(integration_ndarray)


def unpack_integration_results(results):
    return results[:, 0], results[:, 1]


for linked_dataset in all_experiment_data:
    beam_dataset, ecell_dataset = linked_dataset
    beam_id = beam_dataset['exp_id']
    beam_data = beam_dataset['data']
    ecell_id = ecell_dataset['exp_id']
    ecell_data = ecell_dataset['data']
    beam_fit_df = beam_data[ExperimentDataKey.FOM_RESULTS]
    ecell_fit_df = ecell_data[ExperimentDataKey.FOM_RESULTS]

    beam_gamma_integration = integration_to_ndarray(beam_fit_df, integrate_gamma)
    beam_n_integration = integration_to_ndarray(beam_fit_df, integrate_neutrons)
    ecell_gamma_integration = integration_to_ndarray(ecell_fit_df, integrate_gamma)
    ecell_n_integration = integration_to_ndarray(ecell_fit_df, integrate_neutrons)

    
    # dataset_list = [("Beam", beam_dataset), ("Ecell", ecell_dataset)]

    # for i, dataset_list_item in enumerate(dataset_list):
    #     exp_type, dataset = dataset_list_item
    #     exp_id = dataset['exp_id']
    #     exp_data = dataset['data']
        

# gamma_integration = fit_df.apply(
#     lambda row: integrate_gaussian(
#         row["mu1"], row['sigma1'], row['a1']
#     ), axis=1
# )
# neutron_integration = fit_df.apply(
#     lambda row: integrate_gaussian(
#         row["mu2"], row['sigma2'], row['a2']
#     ), axis=1
# )

    beam_g_integral, beam_g_integ_error = unpack_integration_results(beam_gamma_integration)
    beam_n_integral, beam_n_integ_error = unpack_integration_results(beam_n_integration)
    ecell_g_integral, ecell_g_integ_error = unpack_integration_results(ecell_gamma_integration)
    ecell_n_integral, ecell_n_integ_error = unpack_integration_results(ecell_n_integration)

    integral_sums = [
        x.sum() for x in [
            beam_g_integral,
            beam_n_integral,
            ecell_g_integral,
            ecell_n_integral
        ]
    ]
    beam_g_count, beam_n_count, ecell_g_count, ecell_n_count = integral_sums

    error_sums = [
        np.sqrt(np.square(x).sum()) for x in [
            beam_g_integ_error,
            beam_n_integ_error,
            ecell_g_integ_error,
            ecell_n_integ_error
        ]
    ]
    beam_g_count_error, beam_n_count_error, ecell_g_count_error, ecell_n_count_error = error_sums

# gamma_integral, gamma_integ_error = zip(*gamma_integration.tolist())
# gamma_integration = np.array(gamma_integration.tolist())
# gamma_integral = gamma_integration[:, 0]
# gamma_integ_error = gamma_integration[:, 1]
# # print(np.array_str(gamma_integ_error, precision=2, suppress_small=False))
# neutron_integration = np.array(neutron_integration.tolist())
# n_integral = neutron_integration[:, 0]
# n_integ_error = neutron_integration[:, 1]

# gamma_count = gamma_integral.sum()
# gamma_count_error = np.sqrt(np.square(gamma_integ_error).sum())
# neutron_count = n_integral.sum()
# neutron_count_error = np.sqrt(np.square(n_integ_error).sum())
    neutron_delta = ecell_n_count - beam_n_count
    neutron_percent = neutron_delta / beam_n_count
    
    print("-----Datasets-----")
    print(f"Beam:  {beam_id}")
    print(f"Ecell: {ecell_id}")
    print()
    print("---Integrated Counts---")
    print(f"Beam -  Gamma:    {beam_g_count:.2f} +/- {beam_g_count_error:.2e}")
    print(f"        Neutrons: {beam_n_count:.2f} +/- {beam_n_count_error:.2e}")
    print(f"Ecell - Gamma:    {ecell_g_count:.2f} +/- {ecell_g_count_error:.2e}")
    print(f"        Neutrons: {ecell_n_count:.2f} +/- {ecell_n_count_error:.2e}")
    print()
    print("---Neutron Comparison---")
    print(f"Increase: {neutron_delta:.2f}")
    print(f"Percent:  {neutron_percent:.2%}")
# print(f"Gamma:   {gamma_count:.2f} +/- {gamma_count_error:.2e}")
# print(f"Neutron: {neutron_count:.2f} +/- {neutron_count_error:.2e}")